In [ ]:
# 1. Environment & Path Setup
import os
import sys

# Add the project root directory to sys.path so 'pipeline' is importable
sys.path.append(os.path.abspath('../..'))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from sklearn.preprocessing import StandardScaler


In [ ]:
import duckdb

DB_PATH = '/home/kishor/Projects/ShotDist/shotdistributionpredictor/data/events.db'
con = duckdb.connect(DB_PATH, read_only = True)
pm_df = con.execute("SELECT * FROM opspatial WHERE total_reception_events >= 15 AND total_release_events >= 15 AND position != 'Goalkeeper'").df()
con.close()

In [ ]:
pm_df["avg_reception_simpnorm_y"] = np.where(
    pm_df["average_reception_y"] <= 40,
    pm_df["average_reception_y"],
    80 - pm_df["average_reception_y"],
)

pm_df["avg_release_simpnorm_y"] = np.where(
    pm_df["average_release_y"] <= 40,
    pm_df["average_release_y"],
    80 - pm_df["average_release_y"],
)

pm_df["avg_reception_simpnorm_y_abs"] = pm_df["avg_reception_simpnorm_y"].abs()
pm_df["avg_release_simpnorm_y_abs"] = pm_df["avg_release_simpnorm_y"].abs()

feature_cols = [
    "average_reception_x",
    "avg_reception_simpnorm_y_abs",
    "average_release_x",
    "avg_release_simpnorm_y_abs",
]
pm_X = pm_df[feature_cols].dropna()
pm_X_scaled = StandardScaler().fit_transform(pm_X)

pm_k_range = range(2, 11)
pm_inertia = []
pm_silhouette_scores = []

for n_clusters in pm_k_range:
    model = KMeans(n_clusters=n_clusters, n_init=10, random_state=42)
    labels = model.fit_predict(pm_X_scaled)

    pm_inertia.append(model.inertia_)
    pm_silhouette_scores.append(
        silhouette_score(
            pm_X_scaled,
            labels,
            sample_size=min(5000, len(pm_X_scaled)),
            random_state=42,
        )
    )

fig_pm, (ax1_pm, ax2_pm) = plt.subplots(1, 2, figsize=(14, 5))

ax1_pm.plot(list(pm_k_range), pm_inertia, marker="o")
ax1_pm.set(title="Elbow Method", xlabel="Number of Clusters (k)", ylabel="Inertia")

ax2_pm.plot(list(pm_k_range), pm_silhouette_scores, marker="o")
ax2_pm.set(title="Silhouette Scores", xlabel="Number of Clusters (k)", ylabel="Silhouette Score")

plt.tight_layout()
plt.show()

In [ ]:
from mplsoccer import Pitch
from IPython.display import display
from matplotlib.lines import Line2D

pm_X = pm_df[feature_cols].dropna()
pm_X_scaled = StandardScaler().fit_transform(pm_X)
cluster_data = pm_df.loc[pm_X.index].copy()
cluster_counts = []

for n_clusters in [3, 4, 5, 6]:
    model = KMeans(n_clusters=n_clusters, n_init=10, random_state=42)
    cluster_data[f"cluster_k{n_clusters}"] = model.fit_predict(pm_X_scaled)
    pm_df[f"cluster_k{n_clusters}"] = cluster_data[f"cluster_k{n_clusters}"]

    counts = (
        cluster_data.groupby(f"cluster_k{n_clusters}")
        .size()
        .rename("player_match_pair_rows")
        .reset_index()
        .rename(columns={f"cluster_k{n_clusters}": "cluster"})
    )
    counts.insert(0, "k", n_clusters)
    cluster_counts.append(counts)

    pitch = Pitch(pitch_type="statsbomb", pitch_color="white", line_color="#333333")
    fig, ax = pitch.draw(figsize=(11, 7))
    color_map = plt.get_cmap("tab10", n_clusters)
    cluster_col = f"cluster_k{n_clusters}"

    for cluster in range(n_clusters):
        cluster_rows = cluster_data[cluster_data[cluster_col] == cluster]
        color = color_map(cluster)
        pitch.lines(
            cluster_rows["average_reception_x"],
            cluster_rows["average_reception_y"],
            cluster_rows["average_release_x"],
            cluster_rows["average_release_y"],
            color=color,
            alpha=0.25,
            lw=0.7,
            ax=ax,
        )
        pitch.scatter(
            cluster_rows["average_reception_x"],
            cluster_rows["average_reception_y"],
            color=color,
            marker="o",
            s=24,
            ax=ax,
        )
        pitch.scatter(
            cluster_rows["average_release_x"],
            cluster_rows["average_release_y"],
            color=color,
            marker="^",
            s=28,
            ax=ax,
        )

    cluster_handles = [
        Line2D([0], [0], marker="o", color="none", markerfacecolor=color_map(cluster), label=f"Cluster {cluster}")
        for cluster in range(n_clusters)
    ]
    cluster_handles.extend(
        [
            Line2D([0], [0], marker="o", color="#333333", linestyle="none", label="Reception"),
            Line2D([0], [0], marker="^", color="#333333", linestyle="none", label="Release"),
        ]
    )
    ax.legend(handles=cluster_handles, loc="upper left", bbox_to_anchor=(1.01, 1))
    ax.set_title(f"Player-match spatial clusters (k={n_clusters})")
    plt.tight_layout()
    plt.show()

cluster_count_df = pd.concat(cluster_counts, ignore_index=True)
display(cluster_count_df)

In [ ]:
cluster_k3_0_df = pm_df[pm_df["cluster_k3"] == 0].copy()
cluster_k3_1_df = pm_df[pm_df["cluster_k3"] == 1].copy()
cluster_k3_2_df = pm_df[pm_df["cluster_k3"] == 2].copy()

cluster_k4_0_df = pm_df[pm_df["cluster_k4"] == 0].copy()
cluster_k4_1_df = pm_df[pm_df["cluster_k4"] == 1].copy()
cluster_k4_2_df = pm_df[pm_df["cluster_k4"] == 2].copy()
cluster_k4_3_df = pm_df[pm_df["cluster_k4"] == 3].copy()

cluster_k5_0_df = pm_df[pm_df["cluster_k5"] == 0].copy()
cluster_k5_1_df = pm_df[pm_df["cluster_k5"] == 1].copy()
cluster_k5_2_df = pm_df[pm_df["cluster_k5"] == 2].copy()
cluster_k5_3_df = pm_df[pm_df["cluster_k5"] == 3].copy()
cluster_k5_4_df = pm_df[pm_df["cluster_k5"] == 4].copy()

cluster_k6_0_df = pm_df[pm_df["cluster_k6"] == 0].copy()
cluster_k6_1_df = pm_df[pm_df["cluster_k6"] == 1].copy()
cluster_k6_2_df = pm_df[pm_df["cluster_k6"] == 2].copy()
cluster_k6_3_df = pm_df[pm_df["cluster_k6"] == 3].copy()
cluster_k6_4_df = pm_df[pm_df["cluster_k6"] == 4].copy()
cluster_k6_5_df = pm_df[pm_df["cluster_k6"] == 5].copy()

In [ ]:
positions_k3_0 = cluster_k3_0_df["position"].unique()
positions_k3_1 = cluster_k3_1_df["position"].unique()
positions_k3_2 = cluster_k3_2_df["position"].unique()

positions_k4_0 = cluster_k4_0_df["position"].unique()
positions_k4_1 = cluster_k4_1_df["position"].unique()
positions_k4_2 = cluster_k4_2_df["position"].unique()
positions_k4_3 = cluster_k4_3_df["position"].unique()

positions_k5_0 = cluster_k5_0_df["position"].unique()
positions_k5_1 = cluster_k5_1_df["position"].unique()
positions_k5_2 = cluster_k5_2_df["position"].unique()
positions_k5_3 = cluster_k5_3_df["position"].unique()
positions_k5_4 = cluster_k5_4_df["position"].unique()

positions_k6_0 = cluster_k6_0_df["position"].unique()
positions_k6_1 = cluster_k6_1_df["position"].unique()
positions_k6_2 = cluster_k6_2_df["position"].unique()
positions_k6_3 = cluster_k6_3_df["position"].unique()
positions_k6_4 = cluster_k6_4_df["position"].unique()
positions_k6_5 = cluster_k6_5_df["position"].unique()

In [ ]:
position_counts_k3_0 = cluster_k3_0_df["position"].value_counts()
position_counts_k3_1 = cluster_k3_1_df["position"].value_counts()
position_counts_k3_2 = cluster_k3_2_df["position"].value_counts()

position_counts_k4_0 = cluster_k4_0_df["position"].value_counts()
position_counts_k4_1 = cluster_k4_1_df["position"].value_counts()
position_counts_k4_2 = cluster_k4_2_df["position"].value_counts()
position_counts_k4_3 = cluster_k4_3_df["position"].value_counts()

position_counts_k5_0 = cluster_k5_0_df["position"].value_counts()
position_counts_k5_1 = cluster_k5_1_df["position"].value_counts()
position_counts_k5_2 = cluster_k5_2_df["position"].value_counts()
position_counts_k5_3 = cluster_k5_3_df["position"].value_counts()
position_counts_k5_4 = cluster_k5_4_df["position"].value_counts()

position_counts_k6_0 = cluster_k6_0_df["position"].value_counts()
position_counts_k6_1 = cluster_k6_1_df["position"].value_counts()
position_counts_k6_2 = cluster_k6_2_df["position"].value_counts()
position_counts_k6_3 = cluster_k6_3_df["position"].value_counts()
position_counts_k6_4 = cluster_k6_4_df["position"].value_counts()
position_counts_k6_5 = cluster_k6_5_df["position"].value_counts()

In [ ]:
position_counts_df = pd.concat(
    {
        "cluster_k3_0": position_counts_k3_0,
        "cluster_k3_1": position_counts_k3_1,
        "cluster_k3_2": position_counts_k3_2,
        "cluster_k4_0": position_counts_k4_0,
        "cluster_k4_1": position_counts_k4_1,
        "cluster_k4_2": position_counts_k4_2,
        "cluster_k4_3": position_counts_k4_3,
        "cluster_k5_0": position_counts_k5_0,
        "cluster_k5_1": position_counts_k5_1,
        "cluster_k5_2": position_counts_k5_2,
        "cluster_k5_3": position_counts_k5_3,
        "cluster_k5_4": position_counts_k5_4,
        "cluster_k6_0": position_counts_k6_0,
        "cluster_k6_1": position_counts_k6_1,
        "cluster_k6_2": position_counts_k6_2,
        "cluster_k6_3": position_counts_k6_3,
        "cluster_k6_4": position_counts_k6_4,
        "cluster_k6_5": position_counts_k6_5,
    },
    names=["cluster", "position"],
).rename("count").reset_index()

display(position_counts_df)

In [ ]:
position_cluster_counts_df = (
    position_counts_df.pivot(
        index="position",
        columns="cluster",
        values="count",
    )
    .fillna(0)
    .astype(int)
    .reset_index()
)
position_cluster_counts_df.columns.name = None

display(position_cluster_counts_df)

In [ ]:
def autopct_with_counts(values):
    total = values.sum()

    def format_slice(percent):
        count = int(round(percent * total / 100))
        return f"{percent:.1f}%\n(n={count})"

    return format_slice


k_values = [3, 4, 5, 6]
positions = position_cluster_counts_df["position"].tolist()

fig, axes = plt.subplots(
    nrows=len(positions),
    ncols=len(k_values),
    figsize=(20, 4 * len(positions)),
    squeeze=False,
)

cluster_colors = plt.get_cmap("tab10")

for position_index, position in enumerate(positions):
    position_row = position_cluster_counts_df[
        position_cluster_counts_df["position"] == position
    ].iloc[0]

    for k_index, k in enumerate(k_values):
        cluster_cols = [f"cluster_k{k}_{cluster}" for cluster in range(k)]
        counts = position_row[cluster_cols].to_numpy(dtype=int)

        nonzero = counts > 0
        counts_to_plot = counts[nonzero]
        cluster_ids = np.arange(k)[nonzero]
        labels = [f"Cluster {cluster}" for cluster in cluster_ids]
        colors = [cluster_colors(cluster) for cluster in cluster_ids]

        ax = axes[position_index, k_index]
        ax.pie(
            counts_to_plot,
            labels=labels,
            colors=colors,
            autopct=autopct_with_counts(counts_to_plot),
            startangle=90,
            counterclock=False,
            textprops={"fontsize": 8},
        )
        ax.set_title(f"{position} | k={k}")

fig.suptitle("Cluster distribution within each position", y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
plot_data = pm_df.dropna(
    subset=[
        "position",
        "average_reception_x",
        "average_reception_y",
        "average_release_x",
        "average_release_y",
    ]
).copy()

k_values = [3, 4, 5, 6]
positions = sorted(plot_data["position"].unique())
ncols = len(k_values)
nrows = len(positions)

fig, axes = plt.subplots(
    nrows=nrows,
    ncols=ncols,
    figsize=(6 * ncols, 4.5 * nrows),
    squeeze=False,
)

pitch = Pitch(pitch_type="statsbomb", pitch_color="white", line_color="#333333")
cluster_colors = plt.get_cmap("tab10")

for position_index, position in enumerate(positions):
    position_rows = plot_data[plot_data["position"] == position]

    for k_index, k in enumerate(k_values):
        ax = axes[position_index, k_index]
        pitch.draw(ax=ax)

        cluster_col = f"cluster_k{k}"
        for cluster in range(k):
            cluster_rows = position_rows[position_rows[cluster_col] == cluster]
            color = cluster_colors(cluster)

            pitch.lines(
                cluster_rows["average_reception_x"],
                cluster_rows["average_reception_y"],
                cluster_rows["average_release_x"],
                cluster_rows["average_release_y"],
                color=color,
                alpha=0.3,
                lw=0.8,
                ax=ax,
            )
            pitch.scatter(
                cluster_rows["average_reception_x"],
                cluster_rows["average_reception_y"],
                color=color,
                marker="o",
                s=22,
                ax=ax,
            )
            pitch.scatter(
                cluster_rows["average_release_x"],
                cluster_rows["average_release_y"],
                color=color,
                marker="^",
                s=26,
                ax=ax,
            )

        ax.set_title(f"{position} | k={k}")

legend_handles = [
    Line2D(
        [0], [0], marker="o", color="none",
        markerfacecolor=cluster_colors(cluster),
        label=f"Cluster {cluster}",
    )
    for cluster in range(max(k_values))
]
legend_handles.extend([
    Line2D([0], [0], marker="o", color="#333333", linestyle="none", label="Reception"),
    Line2D([0], [0], marker="^", color="#333333", linestyle="none", label="Release"),
])

fig.legend(
    handles=legend_handles,
    loc="upper center",
    bbox_to_anchor=(0.5, 1),
    ncol=4,
)
plt.tight_layout(rect=[0, 0, 1, 0.94])
plt.show()

In [ ]:
plot_data = pm_df[pm_df["team_id"] == 1].dropna(
    subset=[
        "player",
        "average_reception_x",
        "average_reception_y",
        "average_release_x",
        "average_release_y",
    ]
).copy()

k_values = [3, 4, 5, 6]
player_ids = sorted(plot_data["player"].unique(), key=str)
ncols = len(k_values)

fig, axes = plt.subplots(
    nrows=len(player_ids),
    ncols=ncols,
    figsize=(6 * ncols, 4.5 * len(player_ids)),
    squeeze=False,
)

pitch = Pitch(pitch_type="statsbomb", pitch_color="white", line_color="#333333")
cluster_colors = plt.get_cmap("tab10")

for player_index, player_id in enumerate(player_ids):
    player_rows = plot_data[plot_data["player"] == player_id]

    for k_index, k in enumerate(k_values):
        ax = axes[player_index, k_index]
        pitch.draw(ax=ax)

        cluster_col = f"cluster_k{k}"
        for cluster in range(k):
            cluster_rows = player_rows[player_rows[cluster_col] == cluster]
            color = cluster_colors(cluster)

            pitch.lines(
                cluster_rows["average_reception_x"],
                cluster_rows["average_reception_y"],
                cluster_rows["average_release_x"],
                cluster_rows["average_release_y"],
                color=color,
                alpha=0.3,
                lw=0.8,
                ax=ax,
            )
            pitch.scatter(
                cluster_rows["average_reception_x"],
                cluster_rows["average_reception_y"],
                color=color,
                marker="o",
                s=22,
                ax=ax,
            )
            pitch.scatter(
                cluster_rows["average_release_x"],
                cluster_rows["average_release_y"],
                color=color,
                marker="^",
                s=26,
                ax=ax,
            )

        ax.set_title(f"Player {player_id} | k={k}")

legend_handles = [
    Line2D(
        [0], [0], marker="o", color="none",
        markerfacecolor=cluster_colors(cluster),
        label=f"Cluster {cluster}",
    )
    for cluster in range(max(k_values))
]
legend_handles.extend([
    Line2D([0], [0], marker="o", color="#333333", linestyle="none", label="Reception"),
    Line2D([0], [0], marker="^", color="#333333", linestyle="none", label="Release"),
])

fig.legend(
    handles=legend_handles,
    loc="upper center",
    bbox_to_anchor=(0.5, 1),
    ncol=4,
)
plt.tight_layout(rect=[0, 0, 1, 0.94])
plt.show()

In [ ]:
from sklearn.mixture import GaussianMixture
from sklearn.preprocessing import StandardScaler
from mplsoccer import Pitch
from matplotlib.lines import Line2D

# Fit GMMs on rows with valid feature values.
pm_X = (
    pm_df[feature_cols]
    .replace([np.inf, -np.inf], np.nan)
    .dropna()
)
gmm_data = pm_df.loc[pm_X.index].copy()
pm_X_scaled = StandardScaler().fit_transform(pm_X)

# Compare component counts and select the model with the lowest BIC.
component_range = range(2, 11)
gmm_models = {}

for n_components in component_range:
    model = GaussianMixture(
        n_components=n_components,
        covariance_type="full",
        n_init=3,
        random_state=42,
        reg_covar=1e-5,
    )
    model.fit(pm_X_scaled)
    gmm_models[n_components] = model

bic_scores = {
    n_components: model.bic(pm_X_scaled)
    for n_components, model in gmm_models.items()
}
best_n_components = min(bic_scores, key=bic_scores.get)
best_gmm = gmm_models[best_n_components]

gmm_data["gmm_cluster"] = best_gmm.predict(pm_X_scaled)
pm_df.loc[gmm_data.index, "gmm_cluster"] = gmm_data["gmm_cluster"]

print(f"Selected {best_n_components} components (lowest BIC: {bic_scores[best_n_components]:.1f})")

# Plot BIC to compare candidate component counts.
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(list(bic_scores), list(bic_scores.values()), marker="o")
ax.set(
    title="Gaussian Mixture Model selection",
    xlabel="Number of components",
    ylabel="BIC (lower is better)",
)
plt.tight_layout()
plt.show()

# Draw a reproducible sample of unique players.
plot_data = gmm_data.dropna(
    subset=[
        "player",
        "average_reception_x",
        "average_reception_y",
        "average_release_x",
        "average_release_y",
    ]
).copy()

sample_size = min(6, plot_data["player"].nunique())
sample_players = (
    plot_data["player"]
    .drop_duplicates()
    .sample(n=sample_size, random_state=42)
    .tolist()
)

ncols = 2
nrows = (sample_size + ncols - 1) // ncols
fig, axes = plt.subplots(
    nrows=nrows,
    ncols=ncols,
    figsize=(12, 5 * nrows),
    squeeze=False,
)

pitch = Pitch(pitch_type="statsbomb", pitch_color="white", line_color="#333333")
cluster_colors = plt.get_cmap("tab10", best_n_components)

for player_index, player in enumerate(sample_players):
    ax = axes.flat[player_index]
    pitch.draw(ax=ax)
    player_rows = plot_data[plot_data["player"] == player]

    for cluster in range(best_n_components):
        cluster_rows = player_rows[player_rows["gmm_cluster"] == cluster]
        if cluster_rows.empty:
            continue

        color = cluster_colors(cluster)
        pitch.lines(
            cluster_rows["average_reception_x"],
            cluster_rows["average_reception_y"],
            cluster_rows["average_release_x"],
            cluster_rows["average_release_y"],
            color=color,
            alpha=0.35,
            lw=0.8,
            ax=ax,
        )
        pitch.scatter(
            cluster_rows["average_reception_x"],
            cluster_rows["average_reception_y"],
            color=color,
            marker="o",
            s=24,
            ax=ax,
        )
        pitch.scatter(
            cluster_rows["average_release_x"],
            cluster_rows["average_release_y"],
            color=color,
            marker="^",
            s=28,
            ax=ax,
        )

    ax.set_title(str(player))

for ax in axes.flat[sample_size:]:
    ax.set_visible(False)

legend_handles = [
    Line2D(
        [0], [0], marker="o", color="none",
        markerfacecolor=cluster_colors(cluster),
        label=f"Component {cluster}",
    )
    for cluster in range(best_n_components)
]
legend_handles.extend([
    Line2D([0], [0], marker="o", color="#333333", linestyle="none", label="Reception"),
    Line2D([0], [0], marker="^", color="#333333", linestyle="none", label="Release"),
])

fig.legend(
    handles=legend_handles,
    loc="upper center",
    bbox_to_anchor=(0.5, 1),
    ncol=min(best_n_components + 2, 6),
)
fig.suptitle("GMM spatial clusters for sampled players", y=1.03)
plt.tight_layout(rect=[0, 0, 1, 0.95])
plt.show()

In [ ]:
pitch = Pitch(pitch_type="statsbomb", pitch_color="white", line_color="#333333")
fig, ax = pitch.draw(figsize=(13, 8))
cluster_colors = plt.get_cmap("tab10", best_n_components)

for cluster in range(best_n_components):
    cluster_rows = gmm_data[gmm_data["gmm_cluster"] == cluster]
    color = cluster_colors(cluster)

    pitch.lines(
        cluster_rows["average_reception_x"],
        cluster_rows["average_reception_y"],
        cluster_rows["average_release_x"],
        cluster_rows["average_release_y"],
        color=color,
        alpha=0.08,
        lw=0.45,
        ax=ax,
    )
    pitch.scatter(
        cluster_rows["average_reception_x"],
        cluster_rows["average_reception_y"],
        color=color,
        alpha=0.35,
        marker="o",
        s=8,
        ax=ax,
    )
    pitch.scatter(
        cluster_rows["average_release_x"],
        cluster_rows["average_release_y"],
        color=color,
        alpha=0.35,
        marker="^",
        s=10,
        ax=ax,
    )

legend_handles = [
    Line2D(
        [0], [0], marker="o", color="none",
        markerfacecolor=cluster_colors(cluster),
        label=f"Component {cluster}",
    )
    for cluster in range(best_n_components)
]
legend_handles.extend([
    Line2D([0], [0], marker="o", color="#333333", linestyle="none", label="Reception"),
    Line2D([0], [0], marker="^", color="#333333", linestyle="none", label="Release"),
])

ax.legend(handles=legend_handles, loc="upper left", bbox_to_anchor=(1.01, 1))
ax.set_title("GMM spatial clusters across all player-match rows")
plt.tight_layout()
plt.show()

In [ ]:
def gmm_autopct_with_counts(values):
    total = values.sum()

    def format_slice(percent):
        count = int(round(percent * total / 100))
        return f"{percent:.1f}%\n(n={count})"

    return format_slice

position_component_counts = pd.crosstab(
    gmm_data["position"],
    gmm_data["gmm_cluster"],
).reindex(columns=range(best_n_components), fill_value=0)

positions = position_component_counts.index.tolist()
ncols = 3
nrows = (len(positions) + ncols - 1) // ncols
fig, axes = plt.subplots(
    nrows=nrows,
    ncols=ncols,
    figsize=(5 * ncols, 4.5 * nrows),
    squeeze=False,
)
component_colors = plt.get_cmap("tab10", best_n_components)

for index, position in enumerate(positions):
    counts = position_component_counts.loc[position].to_numpy(dtype=int)
    nonzero = counts > 0
    component_ids = np.arange(best_n_components)[nonzero]
    ax = axes.flat[index]
    ax.pie(
        counts[nonzero],
        labels=[f"Component {component}" for component in component_ids],
        colors=[component_colors(component) for component in component_ids],
        autopct=gmm_autopct_with_counts(counts[nonzero]),
        startangle=90,
        counterclock=False,
        textprops={"fontsize": 8},
    )
    ax.set_title(str(position))

for ax in axes.flat[len(positions):]:
    ax.set_visible(False)

fig.suptitle("GMM component distribution by position", y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
position_plot_data = gmm_data.dropna(
    subset=[
        "position",
        "average_reception_x",
        "average_reception_y",
        "average_release_x",
        "average_release_y",
    ]
).copy()
positions = sorted(position_plot_data["position"].unique())
ncols = 2
nrows = (len(positions) + ncols - 1) // ncols
fig, axes = plt.subplots(
    nrows=nrows,
    ncols=ncols,
    figsize=(12 * ncols / 2, 5 * nrows),
    squeeze=False,
)
pitch = Pitch(pitch_type="statsbomb", pitch_color="white", line_color="#333333")
component_colors = plt.get_cmap("tab10", best_n_components)

for index, position in enumerate(positions):
    ax = axes.flat[index]
    pitch.draw(ax=ax)
    position_rows = position_plot_data[position_plot_data["position"] == position]

    for component in range(best_n_components):
        component_rows = position_rows[position_rows["gmm_cluster"] == component]
        if component_rows.empty:
            continue

        color = component_colors(component)
        pitch.lines(
            component_rows["average_reception_x"],
            component_rows["average_reception_y"],
            component_rows["average_release_x"],
            component_rows["average_release_y"],
            color=color,
            alpha=0.3,
            lw=0.7,
            ax=ax,
        )
        pitch.scatter(
            component_rows["average_reception_x"],
            component_rows["average_reception_y"],
            color=color,
            marker="o",
            s=22,
            ax=ax,
        )
        pitch.scatter(
            component_rows["average_release_x"],
            component_rows["average_release_y"],
            color=color,
            marker="^",
            s=26,
            ax=ax,
        )

    ax.set_title(str(position))

for ax in axes.flat[len(positions):]:
    ax.set_visible(False)

component_handles = [
    Line2D(
        [0], [0], marker="o", color="none",
        markerfacecolor=component_colors(component),
        label=f"Component {component}",
    )
    for component in range(best_n_components)
]
component_handles.extend([
    Line2D([0], [0], marker="o", color="#333333", linestyle="none", label="Reception"),
    Line2D([0], [0], marker="^", color="#333333", linestyle="none", label="Release"),
])
fig.legend(handles=component_handles, loc="upper center", bbox_to_anchor=(0.5, 1), ncol=4)
fig.suptitle("GMM spatial components by position", y=1.02)
plt.tight_layout(rect=[0, 0, 1, 0.96])
plt.show()

In [ ]:
team_player_data = gmm_data[gmm_data["team_id"] == 1].dropna(
    subset=[
        "player",
        "average_reception_x",
        "average_reception_y",
        "average_release_x",
        "average_release_y",
    ]
).copy()

players = sorted(team_player_data["player"].unique(), key=str)
ncols = 4
nrows = (len(players) + ncols - 1) // ncols

fig, axes = plt.subplots(
    nrows=nrows,
    ncols=ncols,
    figsize=(6 * ncols, 4.5 * nrows),
    squeeze=False,
)

pitch = Pitch(pitch_type="statsbomb", pitch_color="white", line_color="#333333")
component_colors = plt.get_cmap("tab10", best_n_components)

for index, player in enumerate(players):
    ax = axes.flat[index]
    pitch.draw(ax=ax)
    player_rows = team_player_data[team_player_data["player"] == player]

    for component in range(best_n_components):
        rows = player_rows[player_rows["gmm_cluster"] == component]
        if rows.empty:
            continue

        color = component_colors(component)
        pitch.lines(
            rows["average_reception_x"],
            rows["average_reception_y"],
            rows["average_release_x"],
            rows["average_release_y"],
            color=color,
            alpha=0.3,
            lw=0.7,
            ax=ax,
        )
        pitch.scatter(
            rows["average_reception_x"],
            rows["average_reception_y"],
            color=color,
            marker="o",
            s=22,
            ax=ax,
        )
        pitch.scatter(
            rows["average_release_x"],
            rows["average_release_y"],
            color=color,
            marker="^",
            s=26,
            ax=ax,
        )

    ax.set_title(str(player))

for ax in axes.flat[len(players):]:
    ax.set_visible(False)

handles = [
    Line2D(
        [0], [0], marker="o", color="none",
        markerfacecolor=component_colors(component),
        label=f"Component {component}",
    )
    for component in range(best_n_components)
]
handles.extend([
    Line2D([0], [0], marker="o", color="#333333", linestyle="none", label="Reception"),
    Line2D([0], [0], marker="^", color="#333333", linestyle="none", label="Release"),
])

fig.legend(handles=handles, loc="upper center", bbox_to_anchor=(0.5, 1), ncol=4)
fig.suptitle("GMM spatial components by player (team_id = 1)", y=1.02)
plt.tight_layout(rect=[0, 0, 1, 0.96])
plt.show()

In [ ]:
team_player_data = gmm_data[gmm_data["team_id"] == 1].dropna(
    subset=[
        "player",
        "average_reception_x",
        "average_reception_y",
        "average_release_x",
        "average_release_y",
    ]
).copy()

players = sorted(team_player_data["player"].unique(), key=str)
ncols = 4
nrows = (len(players) + ncols - 1) // ncols

fig, axes = plt.subplots(
    nrows=nrows,
    ncols=ncols,
    figsize=(6 * ncols, 4.5 * nrows),
    squeeze=False,
)

pitch = Pitch(pitch_type="statsbomb", pitch_color="white", line_color="#333333")
component_colors = plt.get_cmap("tab10", best_n_components)

for index, player in enumerate(players):
    ax = axes.flat[index]
    pitch.draw(ax=ax)
    player_rows = team_player_data[team_player_data["player"] == player]

    for component in range(best_n_components):
        rows = player_rows[player_rows["gmm_cluster"] == component]
        if rows.empty:
            continue

        color = component_colors(component)
        pitch.lines(
            rows["average_reception_x"],
            rows["average_reception_y"],
            rows["average_release_x"],
            rows["average_release_y"],
            color=color,
            alpha=0.3,
            lw=0.7,
            ax=ax,
        )
        pitch.scatter(
            rows["average_reception_x"],
            rows["average_reception_y"],
            color=color,
            marker="o",
            s=22,
            ax=ax,
        )
        pitch.scatter(
            rows["average_release_x"],
            rows["average_release_y"],
            color=color,
            marker="^",
            s=26,
            ax=ax,
        )

    ax.set_title(str(player))

for ax in axes.flat[len(players):]:
    ax.set_visible(False)

handles = [
    Line2D(
        [0], [0], marker="o", color="none",
        markerfacecolor=component_colors(component),
        label=f"Component {component}",
    )
    for component in range(best_n_components)
]
handles.extend([
    Line2D([0], [0], marker="o", color="#333333", linestyle="none", label="Reception"),
    Line2D([0], [0], marker="^", color="#333333", linestyle="none", label="Release"),
])

fig.legend(handles=handles, loc="upper center", bbox_to_anchor=(0.5, 1), ncol=4)
fig.suptitle("GMM spatial components by player (team_id = 1)", y=1.02)
plt.tight_layout(rect=[0, 0, 1, 0.96])
plt.show()

In [ ]:
from sklearn.cluster import HDBSCAN
from sklearn.preprocessing import StandardScaler
from mplsoccer import Pitch
from matplotlib.lines import Line2D

pm_X = (
    pm_df[feature_cols]
    .replace([np.inf, -np.inf], np.nan)
    .dropna()
)
hdbscan_data = pm_df.loc[pm_X.index].copy()
pm_X_scaled = StandardScaler().fit_transform(pm_X)

hdbscan_model = HDBSCAN(
    min_cluster_size=30,
    min_samples=10,
    metric="euclidean",
    cluster_selection_method="eom",
)
hdbscan_data["hdbscan_cluster"] = hdbscan_model.fit_predict(pm_X_scaled)
pm_df.loc[hdbscan_data.index, "hdbscan_cluster"] = hdbscan_data["hdbscan_cluster"]

cluster_labels = sorted(hdbscan_data["hdbscan_cluster"].unique())
cluster_colors = plt.get_cmap("tab20", max(1, len(cluster_labels)))
cluster_color_map = {
    label: cluster_colors(index) for index, label in enumerate(cluster_labels)
}
if -1 in cluster_color_map:
    cluster_color_map[-1] = "#777777"

def cluster_name(label):
    return "Noise (-1)" if label == -1 else f"Cluster {label}"

def draw_hdbscan_pitch(ax, rows):
    pitch.draw(ax=ax)
    for label in cluster_labels:
        cluster_rows = rows[rows["hdbscan_cluster"] == label]
        if cluster_rows.empty:
            continue

        color = cluster_color_map[label]
        alpha = 0.05 if label == -1 else 0.25
        pitch.lines(
            cluster_rows["average_reception_x"],
            cluster_rows["average_reception_y"],
            cluster_rows["average_release_x"],
            cluster_rows["average_release_y"],
            color=color,
            alpha=alpha,
            lw=0.6,
            ax=ax,
        )
        pitch.scatter(
            cluster_rows["average_reception_x"],
            cluster_rows["average_reception_y"],
            color=color,
            alpha=max(alpha, 0.15),
            marker="o",
            s=12,
            ax=ax,
        )
        pitch.scatter(
            cluster_rows["average_release_x"],
            cluster_rows["average_release_y"],
            color=color,
            alpha=max(alpha, 0.15),
            marker="^",
            s=14,
            ax=ax,
        )

pitch = Pitch(pitch_type="statsbomb", pitch_color="white", line_color="#333333")

cluster_summary = (
    hdbscan_data["hdbscan_cluster"]
    .value_counts()
    .sort_index()
    .rename_axis("hdbscan_cluster")
    .reset_index(name="row_count")
)
cluster_summary["proportion"] = cluster_summary["row_count"] / len(hdbscan_data)
display(cluster_summary)

In [ ]:
fig, ax = plt.subplots(figsize=(13, 8))
draw_hdbscan_pitch(ax, hdbscan_data)

handles = [
    Line2D(
        [0], [0], marker="o", color="none",
        markerfacecolor=cluster_color_map[label],
        label=cluster_name(label),
    )
    for label in cluster_labels
]
handles.extend([
    Line2D([0], [0], marker="o", color="#333333", linestyle="none", label="Reception"),
    Line2D([0], [0], marker="^", color="#333333", linestyle="none", label="Release"),
])

ax.legend(handles=handles, loc="upper left", bbox_to_anchor=(1.01, 1))
ax.set_title("HDBSCAN clusters across all player-match rows")
plt.tight_layout()
plt.show()

In [ ]:
def hdbscan_autopct(values):
    total = values.sum()

    def format_slice(percent):
        count = int(round(percent * total / 100))
        return f"{percent:.1f}%\n(n={count})"

    return format_slice

position_cluster_counts = pd.crosstab(
    hdbscan_data["position"],
    hdbscan_data["hdbscan_cluster"],
).reindex(columns=cluster_labels, fill_value=0)

positions = position_cluster_counts.index.tolist()
ncols = 3
nrows = (len(positions) + ncols - 1) // ncols

fig, axes = plt.subplots(
    nrows=nrows,
    ncols=ncols,
    figsize=(5 * ncols, 4.5 * nrows),
    squeeze=False,
)

for index, position in enumerate(positions):
    counts = position_cluster_counts.loc[position].to_numpy(dtype=int)
    nonzero = counts > 0
    labels = [cluster_labels[i] for i in np.flatnonzero(nonzero)]

    axes.flat[index].pie(
        counts[nonzero],
        labels=[cluster_name(label) for label in labels],
        colors=[cluster_color_map[label] for label in labels],
        autopct=hdbscan_autopct(counts[nonzero]),
        startangle=90,
        counterclock=False,
        textprops={"fontsize": 8},
    )
    axes.flat[index].set_title(str(position))

for ax in axes.flat[len(positions):]:
    ax.set_visible(False)

fig.suptitle("HDBSCAN cluster distribution by position", y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
position_data = hdbscan_data.dropna(
    subset=[
        "position",
        "average_reception_x",
        "average_reception_y",
        "average_release_x",
        "average_release_y",
    ]
).copy()

positions = sorted(position_data["position"].unique())
ncols = 2
nrows = (len(positions) + ncols - 1) // ncols

fig, axes = plt.subplots(
    nrows=nrows,
    ncols=ncols,
    figsize=(6 * ncols, 5 * nrows),
    squeeze=False,
)

for index, position in enumerate(positions):
    ax = axes.flat[index]
    rows = position_data[position_data["position"] == position]
    draw_hdbscan_pitch(ax, rows)
    ax.set_title(str(position))

for ax in axes.flat[len(positions):]:
    ax.set_visible(False)

handles = [
    Line2D(
        [0], [0], marker="o", color="none",
        markerfacecolor=cluster_color_map[label],
        label=cluster_name(label),
    )
    for label in cluster_labels
]
handles.extend([
    Line2D([0], [0], marker="o", color="#333333", linestyle="none", label="Reception"),
    Line2D([0], [0], marker="^", color="#333333", linestyle="none", label="Release"),
])

fig.legend(handles=handles, loc="upper center", bbox_to_anchor=(0.5, 1), ncol=4)
fig.suptitle("HDBSCAN spatial clusters by position", y=1.02)
plt.tight_layout(rect=[0, 0, 1, 0.96])
plt.show()

In [ ]:
team_player_data = hdbscan_data[hdbscan_data["team_id"] == 1].dropna(
    subset=[
        "player",
        "average_reception_x",
        "average_reception_y",
        "average_release_x",
        "average_release_y",
    ]
).copy()

players = sorted(team_player_data["player"].unique(), key=str)
ncols = 4
nrows = (len(players) + ncols - 1) // ncols

fig, axes = plt.subplots(
    nrows=nrows,
    ncols=ncols,
    figsize=(6 * ncols, 4.5 * nrows),
    squeeze=False,
)

for index, player in enumerate(players):
    ax = axes.flat[index]
    rows = team_player_data[team_player_data["player"] == player]
    draw_hdbscan_pitch(ax, rows)
    ax.set_title(str(player))

for ax in axes.flat[len(players):]:
    ax.set_visible(False)

handles = [
    Line2D(
        [0], [0], marker="o", color="none",
        markerfacecolor=cluster_color_map[label],
        label=cluster_name(label),
    )
    for label in cluster_labels
]
handles.extend([
    Line2D([0], [0], marker="o", color="#333333", linestyle="none", label="Reception"),
    Line2D([0], [0], marker="^", color="#333333", linestyle="none", label="Release"),
])

fig.legend(handles=handles, loc="upper center", bbox_to_anchor=(0.5, 1), ncol=4)
fig.suptitle("HDBSCAN spatial clusters by player (team_id = 1)", y=1.02)
plt.tight_layout(rect=[0, 0, 1, 0.96])
plt.show()